# CPE Minería de Datos — Cuaderno consolidado
## Predicción semanal de demanda por producto — Online Retail

**Objetivo:** construir y comparar tres modelos de regresión para estimar la demanda semanal de productos a partir del historial de ventas.

**Fuente:** UCI Machine Learning Repository — Online Retail  
https://archive.ics.uci.edu/dataset/352/online+retail

Este cuaderno:
- conserva la radiografía ya realizada en Semana 5;
- elimina duplicados en la copia de trabajo;
- identifica y excluye cancelaciones del objetivo de demanda;
- trabaja con ventas positivas;
- separa códigos de producto estándar de códigos administrativos/no-producto;
- elimina semanas parciales de los extremos del periodo;
- construye una serie continua por producto dentro de su periodo activo;
- genera variables históricas sin usar información futura;
- divide entrenamiento/prueba por tiempo;
- compara Regresión Lineal, Árbol de Decisión y Random Forest;
- aplica validación cruzada temporal de 5 particiones;
- calcula MAE, RMSE y R²;
- incluye un baseline ingenuo: demanda de la semana anterior;
- exporta resultados verificables en un ZIP.


**Resultado validado:** Random Forest fue seleccionado como modelo principal por menor MAE y mayor estabilidad temporal; Regresión Lineal obtuvo mejor RMSE y R² en el test final, pero con mayor variabilidad entre folds.

In [ ]:
# 1) Subir el archivo real Online Retail.xlsx
from google.colab import files
uploaded = files.upload()

import os
xlsx_files = [f for f in uploaded.keys() if f.lower().endswith(".xlsx")]
if not xlsx_files:
    raise FileNotFoundError("No se encontró un archivo .xlsx.")
DATA_FILE = xlsx_files[0]
print("Archivo detectado:", DATA_FILE)


In [ ]:
# 2) Importaciones y carga
import json, math, zipfile, warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")

df = pd.read_excel(DATA_FILE)

expected_cols = [
    "InvoiceNo","StockCode","Description","Quantity",
    "InvoiceDate","UnitPrice","CustomerID","Country"
]

print("Dimensión:", df.shape)
print("Columnas:", list(df.columns))

if list(df.columns) != expected_cols:
    raise ValueError("Las columnas no coinciden con la estructura esperada de Online Retail.")

if df.shape[0] != 541909:
    print("ADVERTENCIA: el número de filas no coincide con las 541.909 confirmadas en Semana 5.")
else:
    print("Validación inicial correcta: 541.909 filas.")


## Criterio de preprocesamiento

No se modifica el DataFrame original. Se crea una copia de trabajo.

- **Duplicados:** se eliminan de la copia porque representan repeticiones exactas.
- **Cancelaciones:** UCI documenta que `InvoiceNo` que comienza con `C` indica cancelación.
- **Quantity negativa:** no se considera demanda positiva de venta.
- **CustomerID faltante:** no se elimina por sí solo porque el problema está definido a nivel **producto–semana**, no cliente.
- **Description faltante:** no bloquea el modelado porque el identificador operativo es `StockCode`.
- **Códigos administrativos/no-producto:** se separan usando el patrón habitual de producto de 5 dígitos con letra final opcional. La cantidad excluida se reporta para que la decisión quede documentada.


In [ ]:
# 3) Preprocesamiento documentado
df_work = df.copy()

n_original = len(df_work)
n_duplicates = int(df_work.duplicated().sum())
df_work = df_work.drop_duplicates().copy()
n_after_duplicates = len(df_work)

df_work["InvoiceNo"] = df_work["InvoiceNo"].astype(str)
df_work["StockCode"] = df_work["StockCode"].astype(str)
df_work["InvoiceDate"] = pd.to_datetime(df_work["InvoiceDate"])

df_work["IsCancellation"] = df_work["InvoiceNo"].str.upper().str.startswith("C")
df_work["IsMerchandiseCode"] = df_work["StockCode"].str.fullmatch(r"\d{5}[A-Za-z]?")

cancel_rows = int(df_work["IsCancellation"].sum())
negative_qty_rows = int((df_work["Quantity"] < 0).sum())
zero_qty_rows = int((df_work["Quantity"] == 0).sum())
non_merch_rows = int((~df_work["IsMerchandiseCode"]).sum())

# Demanda = ventas positivas, no canceladas, correspondientes a códigos de producto.
sales = df_work[
    (~df_work["IsCancellation"]) &
    (df_work["Quantity"] > 0) &
    (df_work["IsMerchandiseCode"])
].copy()

print("Filas originales:", n_original)
print("Duplicados eliminados:", n_duplicates)
print("Filas después de duplicados:", n_after_duplicates)
print("Cancelaciones identificadas:", cancel_rows)
print("Filas con Quantity negativa:", negative_qty_rows)
print("Filas con Quantity = 0:", zero_qty_rows)
print("Filas con códigos no-producto separadas:", non_merch_rows)
print("Filas utilizadas como ventas positivas de productos:", len(sales))
print("Productos únicos utilizados:", sales["StockCode"].nunique())


In [ ]:
# 4) Semanas completas y agregación producto-semana
# Semana: lunes a domingo.
sales["WeekStart"] = sales["InvoiceDate"].dt.to_period("W-SUN").dt.start_time

min_date = sales["InvoiceDate"].min().normalize()
max_date = sales["InvoiceDate"].max().normalize()
first_week = sales["WeekStart"].min()
last_week = sales["WeekStart"].max()

# Eliminar semanas incompletas en los extremos del dataset.
first_is_partial = min_date > first_week
last_is_partial = max_date < (last_week + pd.Timedelta(days=6))

complete_sales = sales.copy()
if first_is_partial:
    complete_sales = complete_sales[complete_sales["WeekStart"] != first_week]
if last_is_partial:
    complete_sales = complete_sales[complete_sales["WeekStart"] != last_week]

weekly_obs = (
    complete_sales
    .groupby(["StockCode", "WeekStart"], as_index=False)
    .agg(
        DemandUnits=("Quantity", "sum"),
        InvoiceCount=("InvoiceNo", "nunique")
    )
)

print("Periodo original:", min_date.date(), "a", max_date.date())
print("Primera semana parcial eliminada:", first_is_partial, first_week.date())
print("Última semana parcial eliminada:", last_is_partial, last_week.date())
print("Semanas completas observadas:", weekly_obs["WeekStart"].nunique())
print("Filas producto-semana con venta observada:", len(weekly_obs))


## Serie continua por producto

Una semana sin transacciones **entre la primera y la última semana observada de un producto** se representa con demanda 0.

No se agregan ceros antes de que el producto aparezca por primera vez ni después de su última venta observada, porque el dataset no informa explícitamente si el artículo aún no existía o ya había sido retirado.


In [ ]:
# 5) Construcción de panel continuo dentro del periodo activo de cada producto
parts = []

for stock_code, g in weekly_obs.groupby("StockCode"):
    g = g.sort_values("WeekStart").set_index("WeekStart")
    idx = pd.date_range(g.index.min(), g.index.max(), freq="W-MON")
    h = g.reindex(idx)
    h["StockCode"] = stock_code
    h["DemandUnits"] = h["DemandUnits"].fillna(0)
    h["InvoiceCount"] = h["InvoiceCount"].fillna(0)
    h.index.name = "WeekStart"
    parts.append(h.reset_index())

panel = pd.concat(parts, ignore_index=True)
panel = panel.sort_values(["StockCode", "WeekStart"]).reset_index(drop=True)

print("Filas del panel producto-semana:", len(panel))
print("Productos en panel:", panel["StockCode"].nunique())
print("Semanas globales:", panel["WeekStart"].nunique())
print("Semanas internas sin venta representadas como 0:",
      int((panel["DemandUnits"] == 0).sum()))


In [ ]:
# 6) Feature engineering SIN fuga de información
grp = panel.groupby("StockCode", group_keys=False)

panel["DemandLag1"] = grp["DemandUnits"].shift(1)
panel["DemandLag2"] = grp["DemandUnits"].shift(2)
panel["DemandRolling4"] = (
    grp["DemandUnits"]
    .apply(lambda s: s.shift(1).rolling(4, min_periods=4).mean())
    .reset_index(level=0, drop=True)
)
panel["InvoiceCountLag1"] = grp["InvoiceCount"].shift(1)

iso_week = panel["WeekStart"].dt.isocalendar().week.astype(int)
panel["WeekSin"] = np.sin(2 * np.pi * iso_week / 52.0)
panel["WeekCos"] = np.cos(2 * np.pi * iso_week / 52.0)

features = [
    "DemandLag1",
    "DemandLag2",
    "DemandRolling4",
    "InvoiceCountLag1",
    "WeekSin",
    "WeekCos"
]
target = "DemandUnits"

model_df = panel.dropna(subset=features + [target]).copy()
model_df = model_df.sort_values(["WeekStart", "StockCode"]).reset_index(drop=True)

print("Filas listas para modelar:", len(model_df))
print("Productos modelables:", model_df["StockCode"].nunique())
print("Semanas modelables:", model_df["WeekStart"].nunique())
print("Variables predictoras:", features)


## División temporal

No se usa `train_test_split` aleatorio.

Las semanas más antiguas se utilizan para entrenamiento y las últimas para prueba. Esto evita que el modelo aprenda con información cronológicamente posterior al periodo que intenta predecir.


In [ ]:
# 7) División temporal 80/20 por semanas completas
weeks = np.array(sorted(model_df["WeekStart"].unique()))
cut_idx = int(np.floor(len(weeks) * 0.80))
cut_idx = min(max(cut_idx, 1), len(weeks) - 1)

train_weeks = weeks[:cut_idx]
test_weeks = weeks[cut_idx:]

train = model_df[model_df["WeekStart"].isin(train_weeks)].copy()
test = model_df[model_df["WeekStart"].isin(test_weeks)].copy()

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print("Semanas entrenamiento:", len(train_weeks))
print("Semanas prueba:", len(test_weeks))
print("Última semana de entrenamiento:", pd.Timestamp(train_weeks[-1]).date())
print("Primera semana de prueba:", pd.Timestamp(test_weeks[0]).date())
print("Filas entrenamiento:", len(train))
print("Filas prueba:", len(test))


## Modelos

1. **Regresión Lineal:** referencia interpretable y simple.
2. **Árbol de Decisión:** captura relaciones no lineales; se limita la profundidad para reducir sobreajuste.
3. **Random Forest:** ensamble de árboles para mejorar estabilidad.

También se incluye un **baseline ingenuo** (`predicción = demanda de la semana anterior`). No cuenta como una de las tres técnicas; sirve como control para comprobar si los modelos realmente aportan valor.


In [ ]:
# 8) Modelos y evaluación en test
models = {
    "Regresion_Lineal": LinearRegression(),
    "Arbol_Decision": DecisionTreeRegressor(
        max_depth=10,
        min_samples_leaf=10,
        random_state=42
    ),
    "Random_Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
}

def metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }

test_results = []

# Baseline
baseline_pred = test["DemandLag1"].to_numpy()
base_m = metrics(y_test, baseline_pred)
test_results.append({"Modelo": "Baseline_Lag1", **base_m})

predictions = test[["StockCode", "WeekStart", "DemandUnits"]].copy()
predictions["Pred_Baseline_Lag1"] = baseline_pred

fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.maximum(model.predict(X_test), 0)  # demanda no negativa
    m = metrics(y_test, pred)
    test_results.append({"Modelo": name, **m})
    predictions[f"Pred_{name}"] = pred
    fitted_models[name] = model

test_metrics = pd.DataFrame(test_results).sort_values("MAE").reset_index(drop=True)
print(test_metrics)


## Validación cruzada temporal

La validación cruzada se realiza sobre **semanas**, no sobre filas sueltas.  
Cada fold usa un bloque temporal anterior para entrenar y un bloque posterior para validar, evitando mezclar productos de una misma semana entre entrenamiento y validación.


In [ ]:
# 9) Validación cruzada temporal de 5 particiones sobre semanas de entrenamiento
unique_train_weeks = np.array(sorted(train["WeekStart"].unique()))

if len(unique_train_weeks) < 7:
    raise ValueError("No hay suficientes semanas para una validación temporal de 5 particiones.")

tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(unique_train_weeks), start=1):
    fold_train_weeks = unique_train_weeks[tr_idx]
    fold_val_weeks = unique_train_weeks[val_idx]

    fold_train = train[train["WeekStart"].isin(fold_train_weeks)]
    fold_val = train[train["WeekStart"].isin(fold_val_weeks)]

    Xtr, ytr = fold_train[features], fold_train[target]
    Xv, yv = fold_val[features], fold_val[target]

    for name, template in models.items():
        mdl = clone(template)
        mdl.fit(Xtr, ytr)
        pv = np.maximum(mdl.predict(Xv), 0)
        m = metrics(yv, pv)
        cv_rows.append({
            "Fold": fold,
            "Modelo": name,
            "Train_Week_Start": pd.Timestamp(fold_train_weeks[0]),
            "Train_Week_End": pd.Timestamp(fold_train_weeks[-1]),
            "Val_Week_Start": pd.Timestamp(fold_val_weeks[0]),
            "Val_Week_End": pd.Timestamp(fold_val_weeks[-1]),
            **m
        })

cv_results = pd.DataFrame(cv_rows)

cv_summary = (
    cv_results
    .groupby("Modelo")
    .agg(
        CV_MAE_Mean=("MAE", "mean"),
        CV_MAE_Std=("MAE", "std"),
        CV_RMSE_Mean=("RMSE", "mean"),
        CV_RMSE_Std=("RMSE", "std"),
        CV_R2_Mean=("R2", "mean"),
        CV_R2_Std=("R2", "std"),
    )
    .reset_index()
    .sort_values("CV_MAE_Mean")
)

print(cv_summary)


In [ ]:
# 10) Comparación consolidada
comparison = test_metrics.merge(cv_summary, on="Modelo", how="left")
print(comparison)

best_model_name = (
    comparison[comparison["Modelo"] != "Baseline_Lag1"]
    .sort_values(["MAE", "RMSE"])
    .iloc[0]["Modelo"]
)

print("\nMejor modelo por MAE/RMSE de prueba:", best_model_name)

# Auditoría básica de resultados sospechosamente perfectos
best_row = comparison[comparison["Modelo"] == best_model_name].iloc[0]
if best_row["R2"] > 0.995 or best_row["MAE"] < 1e-6:
    print("ALERTA: resultado casi perfecto. Revisar posible fuga de información o estructura del target.")
else:
    print("No se detecta una métrica de prueba artificialmente perfecta.")


In [ ]:
# 11) Gráfico 1: MAE y RMSE por modelo
plot_df = test_metrics.set_index("Modelo")[["MAE", "RMSE"]]
ax = plot_df.plot(kind="bar", figsize=(10,5))
ax.set_title("Comparación de error en el conjunto de prueba")
ax.set_ylabel("Unidades")
ax.set_xlabel("Modelo")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("comparacion_mae_rmse.png", dpi=160)
plt.show()


In [ ]:
# 12) Gráfico 2: demanda agregada real vs predicha por semana
weekly_pred = (
    predictions
    .groupby("WeekStart", as_index=False)
    .sum(numeric_only=True)
)

plt.figure(figsize=(12,6))
plt.plot(weekly_pred["WeekStart"], weekly_pred["DemandUnits"], label="Real", linewidth=2)
for name in models.keys():
    plt.plot(
        weekly_pred["WeekStart"],
        weekly_pred[f"Pred_{name}"],
        label=name
    )
plt.title("Demanda agregada real vs predicha — semanas de prueba")
plt.xlabel("Semana")
plt.ylabel("Unidades")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("demanda_semanal_test.png", dpi=160)
plt.show()


In [ ]:
# 13) Gráfico 3: importancia de variables de Random Forest
rf = fitted_models["Random_Forest"]
importance = (
    pd.DataFrame({
        "Variable": features,
        "Importancia": rf.feature_importances_
    })
    .sort_values("Importancia", ascending=False)
)

ax = importance.set_index("Variable").plot(kind="bar", figsize=(9,5), legend=False)
ax.set_title("Importancia de variables — Random Forest")
ax.set_ylabel("Importancia")
ax.set_xlabel("Variable")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("importancia_random_forest.png", dpi=160)
plt.show()

print(importance)


In [ ]:
# 14) Exportar resultados verificables
out = Path("resultados_paso6")
out.mkdir(exist_ok=True)

test_metrics.to_csv(out / "metricas_test.csv", index=False)
cv_results.to_csv(out / "validacion_cruzada_folds.csv", index=False)
cv_summary.to_csv(out / "validacion_cruzada_resumen.csv", index=False)
comparison.to_csv(out / "comparacion_modelos.csv", index=False)
importance.to_csv(out / "importancia_random_forest.csv", index=False)
predictions.to_csv(out / "predicciones_test.csv", index=False)
model_df.to_csv(out / "dataset_producto_semana_modelado.csv", index=False)

# Exportar el modelo Random Forest REAL entrenado en esta ejecución
model_path = out / "modelo_random_forest_demanda.pkl"
joblib.dump(fitted_models["Random_Forest"], model_path)
print("Modelo exportado:", model_path)

for img in [
    "comparacion_mae_rmse.png",
    "demanda_semanal_test.png",
    "importancia_random_forest.png"
]:
    Path(img).replace(out / img)

summary = {
    "archivo_origen": DATA_FILE,
    "filas_originales": int(n_original),
    "duplicados_eliminados": int(n_duplicates),
    "filas_post_duplicados": int(n_after_duplicates),
    "cancelaciones_identificadas": int(cancel_rows),
    "quantity_negativa": int(negative_qty_rows),
    "codigos_no_producto_separados": int(non_merch_rows),
    "ventas_positivas_producto": int(len(sales)),
    "productos_modelables": int(model_df["StockCode"].nunique()),
    "semanas_modelables": int(model_df["WeekStart"].nunique()),
    "semanas_entrenamiento": int(len(train_weeks)),
    "semanas_prueba": int(len(test_weeks)),
    "features": features,
    "target": target,
    "mejor_modelo_test": best_model_name,
    "modelos": {k: str(v) for k, v in models.items()},
}

with open(out / "resumen_paso6.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

zip_name = "resultados_paso6.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob("*"):
        z.write(p, arcname=p.relative_to(out))

print("Archivos generados en:", out.resolve())
print("ZIP generado:", zip_name)
print("\nEl ZIP ahora incluye modelo_random_forest_demanda.pkl. Descárguelo y úselo en la API/web app del repositorio.")


## Criterio para interpretar el resultado

El modelo final **no se elige solo por R²**.

Se revisa:
1. menor MAE;
2. menor RMSE;
3. R² razonable;
4. estabilidad entre los 5 folds;
5. comparación con el baseline `DemandLag1`;
6. ausencia de resultados artificialmente perfectos;
7. interpretación empresarial del error.

Si los modelos no superan claramente al baseline, eso también es un resultado válido y debe explicarse: significa que el historial inmediato puede contener tanta información como los modelos construidos con estas variables.


## Interpretación final de la ejecución validada

| Modelo | MAE | RMSE | R² |
|---|---:|---:|---:|
| Random Forest | 38.88 | 134.34 | 0.344 |
| Árbol de Decisión | 39.93 | 142.64 | 0.261 |
| Baseline Lag1 | 40.89 | 149.59 | 0.187 |
| Regresión Lineal | 40.92 | 130.31 | 0.383 |

Random Forest presenta el menor MAE y una validación cruzada temporal estable (CV-MAE 25.84 ± 2.32; CV-R² 0.344 ± 0.045). Regresión Lineal presenta mejor RMSE y R² en el conjunto de prueba, pero su variabilidad entre particiones es sustancialmente mayor. Por ello, Random Forest se conserva como modelo principal por consistencia y Regresión Lineal como alternativa relevante.

La variable `DemandRolling4` aparece como la más importante dentro de Random Forest, lo que indica que el promedio reciente de cuatro semanas concentra una parte importante de la señal predictiva.

**Limitación:** la variable objetivo representa unidades vendidas observadas, no demanda latente no satisfecha por falta de inventario.

## Capa de explotación del modelo

La ejecución de este cuaderno exporta `modelo_random_forest_demanda.pkl`, que corresponde al objeto `fitted_models["Random_Forest"]` entrenado con los datos reales del proyecto. Ese archivo se utiliza posteriormente desde una API FastAPI y una aplicación web mínima.

La API recibe `DemandLag1`, `DemandLag2`, `DemandRolling4`, `InvoiceCountLag1` y el número de semana; calcula internamente `WeekSin` y `WeekCos` y devuelve la demanda semanal estimada.
